# CacheGen TTFT Reproduction on Colab

This notebook runs the same `ttft_benchmark.py` core used by `run_benchmarks.py`. The default configuration targets the single-A100 claim run for `mistralai/Mistral-7B-Instruct-v0.3` at `3000 Mbps` with `quantized_fp8` vs `cachegen`.

In [ ]:
%%bash
set -euxo pipefail
cd /content
python -m pip install -q uv
if [ ! -d cachegen ]; then
  git clone --recurse-submodules https://github.com/Khiem17204/cachegen.git
fi
cd /content/cachegen
git submodule update --init --recursive third_party/vllm
uv pip install -r requirements.txt
VLLM_USE_PRECOMPILED=1 uv pip install --editable ./third_party/vllm --torch-backend=auto


In [ ]:
%cd /content/cachegen
import json
from ttft_benchmark import capture_environment_meta

print(json.dumps(capture_environment_meta(), indent=2))


In [ ]:
%cd /content/cachegen
from pathlib import Path
from ttft_benchmark import BenchmarkConfig, ModelConfig

config = BenchmarkConfig(
    models=[
        ModelConfig(
            model="mistralai/Mistral-7B-Instruct-v0.3",
            max_model_len=8192,
            dtype="bfloat16",
        )
    ],
    modes=["quantized_fp8", "cachegen"],
    bandwidth_mbps=[3000.0],
    prompt_lengths=[2048, 4096],
    repeats=5,
    max_tokens=16,
    gpu_memory_utilization=0.85,
    cache_root=Path("/content/cachegen/benchmarks/cache_store"),
    output_path=Path("/content/cachegen/benchmarks/results.json"),
)
config


In [ ]:
%cd /content/cachegen
import json
from ttft_benchmark import run_benchmark_suite, save_report

report = await run_benchmark_suite(config)
save_report(report, config.output_path)
print(json.dumps(report["claim_check"], indent=2))


In [ ]:
%cd /content/cachegen
from IPython.display import Image, display
from visualize_results import main as render_plots

render_plots(config.output_path)
display(Image(filename="/content/cachegen/benchmarks/ttft_comparison_3gbps.png"))
display(Image(filename="/content/cachegen/benchmarks/transport_breakdown_3gbps.png"))


In [ ]:
# Optional: only copy artifacts to Drive after timing is complete.
from pathlib import Path
import shutil

drive_root = Path("/content/drive/MyDrive/cachegen-ttft-artifacts")
if drive_root.parent.exists():
    drive_root.mkdir(parents=True, exist_ok=True)
    for artifact_name in [
        "results.json",
        "ttft_comparison_3gbps.png",
        "transport_breakdown_3gbps.png",
    ]:
        shutil.copy2(Path("/content/cachegen/benchmarks") / artifact_name, drive_root / artifact_name)
    print(f"Copied artifacts to {drive_root}")
else:
    print("Mount Google Drive first if you want to export artifacts.")
